# B.1 — SFT baseline vs v2 GRPO  
**GPU: A100 40GB** &nbsp;·&nbsp; **Cost: ~10 Colab units** &nbsp;·&nbsp; **Runtime: ~50 min**

Trains a SFT (supervised fine-tuning) baseline on the **same training corpus** v2 GRPO used, with **identical LoRA hyperparams** (r=32, alpha=64, same target modules), then evaluates it on the same bench. Gives you a measured answer to the canonical judge question:

> *"Did GRPO actually help, or could plain SFT have given you the same numbers?"*

## Outputs
- `checkpoints/sft_baseline_lora/` — SFT LoRA adapter (~50 MB)
- `logs/eval_sft.json` — bench eval (n=174, detection / FPR / F1 / per-difficulty)
- `logs/sft_vs_grpo_comparison.json` — side-by-side SFT vs v2 GRPO numbers

## Run order (same as B.5)
1. `Runtime → Disconnect and delete runtime`, then connect with **GPU A100** (or L4 24GB if A100 unavailable).
2. Run cells 1, 2, 3 in order. Cell 3 will kill the kernel — wait ~10 sec for auto-reconnect.
3. Re-run Cell 2 (clone, idempotent), **SKIP Cell 3**, run Cells 4 onwards normally.

In [ ]:
# CELL 1: GPU check (A100 preferred; L4/T4 acceptable but slower)
import subprocess, sys
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('Python:', sys.version, '\nGPU:', out)
assert any(g in out for g in ('A100', 'L4', 'T4')), f'Need GPU runtime; got: {out}'
if 'A100' not in out:
    print('\nWARNING: A100 not detected. Training will work on L4/T4 but be ~3-5x slower.')
    print('Recommended: switch to A100 via Runtime → Change runtime type.')

In [ ]:
# CELL 2: Clone repo (idempotent)
import os
from pathlib import Path
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh'
REPO_DIR = '/content/Chakravyuh'
if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase --autostash
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
# CELL 3: Install + KERNEL RESTART (proven pattern from B.5; adds trl + datasets for SFT)
import os, time
REPO_DIR = '/content/Chakravyuh'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/UjjwalPardeshi/Chakravyuh {REPO_DIR}
os.chdir(REPO_DIR)

# 3a. Wipe Colab's broken torch+torchvision
!pip uninstall -y -q torch torchvision torchaudio 2>&1 | tail -2

# 3b. Matched cu121 torch+torchvision pair
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
    'torch==2.5.1+cu121' 'torchvision==0.20.1+cu121' 2>&1 | tail -3

# 3c. Chakravyuh + base deps
!pip install -q -e {REPO_DIR} --no-deps 2>&1 | tail -2
!pip install -q --no-deps \
    'pydantic>=2.6' 'python-dotenv>=1.0' 'jsonlines>=4.0' \
    'openenv-core>=0.2.3,<0.3' 'fastapi>=0.115' 'uvicorn>=0.30' 'tqdm' \
    'transformers==4.46.3' 'peft>=0.14,<0.20' 'accelerate==1.0.1' 'bitsandbytes==0.44.1' \
    'tokenizers>=0.20,<0.21' 'huggingface-hub>=0.24,<0.27' \
    'sentencepiece' 'safetensors' \
    2>&1 | tail -3

# 3d. SFT-specific: trl + datasets (these need their deps; install with deps but pin)
!pip install -q --no-cache-dir 'trl==0.12.2' 'datasets>=2.19,<3.0' 'numpy<2' 2>&1 | tail -3

# 3e. fastmcp + mcp (transitive openenv-core deps)
!pip install -q 'fastmcp>=0.4' 'mcp>=1.0' 'httpx' 'anyio' 2>&1 | tail -3

# 3f. Verify
!python -c "import torch, torchvision; print(f'torch: {torch.__version__} | torchvision: {torchvision.__version__}')"
!python -c "from transformers.cache_utils import SlidingWindowCache; print('SlidingWindowCache: OK')"
!python -c "from trl import SFTTrainer, SFTConfig; print('trl SFTTrainer: OK')"
!python -c "import openenv; print('openenv: import OK')" 2>&1 | tail -2

# 3g. Kill kernel
print('\n' + '=' * 60)
print('SETUP COMPLETE. Kernel restarting in 2s...')
print('After restart: re-run Cell 2 (clone), SKIP Cell 3, run Cells 4 onwards.')
print('=' * 60)
time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
# CELL 4: Build SFT dataset from the same template files v2 GRPO used
import sys, json
from pathlib import Path
sys.path.insert(0, REPO_DIR)

from chakravyuh_env.agents.llm_analyzer import DEFAULT_SYSTEM_PROMPT, DEFAULT_USER_PROMPT_TEMPLATE
from training.grpo_analyzer import _flat_template_to_prompt, _multiturn_to_prompt, _load_json_templates

ENV_DIR = Path(REPO_DIR) / 'chakravyuh_env'

scam_files = ['scammer_templates.json', 'augmented_templates.json', 'paraphrase_templates.json',
              'regional_templates.json', 'scam_novel.json']
benign_files = ['benign_templates.json', 'benign_augmented.json', 'benign_augmented_v2.json']
multiturn_files = ['multiturn_templates.json']

examples = []

def make_example(scammer_text: str, is_scam: bool, signals: list[str] | None = None):
    """Construct one (prompt, gold_answer) chat-style example."""
    if not scammer_text.strip():
        return None
    chat_body = '\n'.join(f'- {line}' for line in scammer_text.split('\n') if line.strip())
    user_msg = DEFAULT_USER_PROMPT_TEMPLATE.format(chat=chat_body)
    if is_scam:
        sigs = signals or []
        score = 0.95
        explanation = f"Scam pattern detected: {', '.join(sigs[:3]) if sigs else 'unsolicited info request'}."
    else:
        sigs = []
        score = 0.05
        explanation = 'Legitimate message — no scam signals detected.'
    gold_answer = json.dumps({'score': score, 'signals': sigs, 'explanation': explanation})
    return {
        'messages': [
            {'role': 'system', 'content': DEFAULT_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg},
            {'role': 'assistant', 'content': gold_answer},
        ]
    }

# Scam templates (single-turn)
for fname in scam_files:
    tpls = _load_json_templates(ENV_DIR / fname)
    for t in tpls:
        prompt, signals = _flat_template_to_prompt(t)
        ex = make_example(prompt, is_scam=True, signals=signals)
        if ex: examples.append(ex)

# Multi-turn scam templates
for fname in multiturn_files:
    tpls = _load_json_templates(ENV_DIR / fname)
    for t in tpls:
        prompt, signals = _multiturn_to_prompt(t)
        ex = make_example(prompt, is_scam=True, signals=signals)
        if ex: examples.append(ex)

# Benign templates
for fname in benign_files:
    tpls = _load_json_templates(ENV_DIR / fname)
    for t in tpls:
        text = t.get('text') or t.get('message') or ''
        ex = make_example(text, is_scam=False)
        if ex: examples.append(ex)

scam_count = sum(1 for e in examples if '"score": 0.95' in e['messages'][-1]['content'])
benign_count = len(examples) - scam_count
print(f'Built {len(examples)} SFT examples ({scam_count} scam + {benign_count} benign).')
print(f'\nExample (first scam):')
print(json.dumps(examples[0], indent=2)[:600])

In [ ]:
# CELL 5: Load Qwen2.5-7B + LoRA (matching v2 GRPO hyperparams exactly)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'right'  # SFT requires right-padding (Qwen default is 'left' for generation)
assert tokenizer.chat_template, 'Tokenizer missing chat_template — SFT messages format will fail'

print('Loading base model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Match v2 GRPO LoRA config exactly (r=32, alpha=64, all 7 target modules)
lora_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# CELL 6: Configure SFT training (~30-40 min on A100)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ds = Dataset.from_list(examples)
print(f'Dataset size: {len(ds)}')

OUTPUT_DIR = f'{REPO_DIR}/checkpoints/sft_baseline_lora'

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    seed=42,
    max_seq_length=1024,
    packing=False,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=sft_config,
    tokenizer=tokenizer,
)
print('SFTTrainer ready. Starting training...')

In [ ]:
# CELL 7: TRAIN (the main event)
try:
    trainer.train()
except torch.cuda.OutOfMemoryError as e:
    print('\nCUDA OOM during training. Likely fixes:')
    print('  1. Switch runtime to A100 (you may be on L4/T4)')
    print('  2. In Cell 6, reduce per_device_train_batch_size 4 -> 1 + raise gradient_accumulation_steps 2 -> 8')
    print('  3. Reduce max_seq_length 1024 -> 512')
    raise
trainer.save_model(f'{OUTPUT_DIR}/final')
print(f'\nTraining complete. LoRA saved to: {OUTPUT_DIR}/final')
!ls -la {OUTPUT_DIR}/final/

In [ ]:
# CELL 8: Free training memory + reload model with SFT LoRA for eval
import gc, torch
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print('Training memory freed.')

from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer

sft_analyzer = LLMAnalyzer(
    model_name=BASE_MODEL,
    lora_path=f'{OUTPUT_DIR}/final',
    use_unsloth=False,
    load_in_4bit=True,
    max_new_tokens=200,
    temperature=0.0,
    flag_threshold=0.55,
    seed=42,
)
sft_analyzer.load()

# Sanity check
smoke_scam = 'URGENT: Your SBI account will be blocked today. Share OTP 234567 to verify KYC.'
smoke_benign = 'Your OTP for HDFC login is 563182. Valid 5 min. Do NOT share with anyone.'
print(f'Scam test  : {sft_analyzer.score_text(smoke_scam):.3f}  (expected >= 0.7)')
print(f'Benign test: {sft_analyzer.score_text(smoke_benign):.3f}  (expected <= 0.3)')

In [ ]:
# CELL 9: Eval SFT analyzer on the 174-scenario bench
from tqdm.auto import tqdm
import time
from collections import defaultdict

BENCH = Path(REPO_DIR) / 'data' / 'chakravyuh-bench-v0' / 'scenarios.jsonl'
with BENCH.open() as f:
    scenarios = [json.loads(l) for l in f if l.strip()]
print(f'Bench: {len(scenarios)} scenarios')

THR = 0.55
per_row, errors = [], []
t0 = time.time()
for s in tqdm(scenarios, desc='SFT eval'):
    text = ' '.join(m['text'] for m in s['attack_sequence'] if m['sender'] == 'scammer')
    try:
        score = float(sft_analyzer.score_text(text))
    except Exception as e:
        errors.append({'scenario_id': s['id'], 'error': str(e)})
        continue
    per_row.append({
        'scenario_id': s['id'],
        'score': score,
        'predicted_flag': bool(score >= THR),
        'ground_truth': bool(s['ground_truth']['is_scam']),
        'difficulty': s['ground_truth']['difficulty'],
    })
elapsed = time.time() - t0
print(f'\nScored {len(per_row)} / {len(scenarios)} in {elapsed/60:.1f} min')
if errors: print('Errors:', errors[:3])

def aggregate(rows):
    n = len(rows)
    if not n: return {'n': 0}
    tp = sum(1 for r in rows if r['ground_truth'] and r['predicted_flag'])
    fn = sum(1 for r in rows if r['ground_truth'] and not r['predicted_flag'])
    fp = sum(1 for r in rows if not r['ground_truth'] and r['predicted_flag'])
    tn = sum(1 for r in rows if not r['ground_truth'] and not r['predicted_flag'])
    scam, benign = tp + fn, fp + tn
    det = tp / scam if scam else 0.0
    fpr = fp / benign if benign else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (2*prec*det/(prec+det)) if (prec+det) else 0.0
    return {'n': n, 'n_scam': scam, 'n_benign': benign, 'detection': det, 'fpr': fpr, 'precision': prec, 'f1': f1}

sft_overall = aggregate(per_row)
by_diff = defaultdict(list)
for r in per_row: by_diff[r['difficulty']].append(r)
sft_by_diff = {d: aggregate(rs) for d, rs in by_diff.items()}

print(f"\n=== SFT baseline (n={sft_overall['n']}) ===")
print(f"detection: {sft_overall['detection']:.1%} | FPR: {sft_overall['fpr']:.1%} | F1: {sft_overall['f1']:.4f}")
for d in ('easy', 'medium', 'hard', 'novel'):
    if d in sft_by_diff:
        m = sft_by_diff[d]
        print(f"  {d:8s} n={m['n']:3d}  det={m['detection']:.1%}  fpr={m['fpr']:.1%}")

In [ ]:
# CELL 10: Side-by-side SFT vs v2 GRPO comparison
# Read v2 GRPO numbers from logs/eval_v2.json (single source of truth)
_v2_path = Path(REPO_DIR) / 'logs' / 'eval_v2.json'
_v2 = json.loads(_v2_path.read_text())['lora_v2']
GRPO_NUMBERS = {
    'detection': _v2['detection'],
    'fpr': _v2['fpr'],
    'precision': _v2['precision'],
    'f1': _v2['f1'],
    'n': _v2['n'],
    'per_difficulty': {d: {'n': v['n'], 'detection': v['detection_rate']} for d, v in _v2['per_difficulty'].items()},
}
print(f'Loaded v2 GRPO numbers from {_v2_path.name}: det={GRPO_NUMBERS["detection"]:.1%}, FPR={GRPO_NUMBERS["fpr"]:.1%}')

print('=' * 70)
print(f"{'Metric':<20s} {'v2 GRPO':>15s} {'SFT baseline':>15s} {'Δ (SFT−GRPO)':>15s}")
print('-' * 70)
for k in ('detection', 'fpr', 'precision', 'f1'):
    g = GRPO_NUMBERS[k]
    s = sft_overall[k]
    delta = s - g
    fmt = '.1%' if k != 'f1' else '.4f'
    print(f"{k:<20s} {g:>15{fmt}} {s:>15{fmt}} {delta:>+15{fmt}}")
print('=' * 70)
print('\nPer-difficulty detection:')
print(f"{'Difficulty':<12s} {'v2 GRPO':>10s} {'SFT':>10s}")
for d in ('easy', 'medium', 'hard', 'novel'):
    g = GRPO_NUMBERS['per_difficulty'].get(d, {})
    s = sft_by_diff.get(d, {})
    if g and s:
        print(f"  {d:<10s} {g['detection']:>10.1%} {s['detection']:>10.1%}")

print('\n\nNarrative interpretation:')
fpr_ratio = sft_overall['fpr'] / GRPO_NUMBERS['fpr'] if GRPO_NUMBERS['fpr'] > 0 else float('inf')
det_delta = sft_overall['detection'] - GRPO_NUMBERS['detection']
if abs(det_delta) < 0.02 and fpr_ratio < 1.5:
    print("  SFT TIES GRPO. Headline: 'Env design is the contribution; training algo less load-bearing.'")
elif sft_overall['fpr'] > GRPO_NUMBERS['fpr'] * 1.5:
    print(f"  GRPO WINS on FPR ({sft_overall['fpr']:.1%} SFT vs {GRPO_NUMBERS['fpr']:.1%} GRPO).")
    print("  Headline: 'GRPO's reward shaping decisively reduces over-flagging vs imitation learning.'")
elif sft_overall['detection'] < GRPO_NUMBERS['detection'] - 0.05:
    print("  GRPO WINS on detection. Headline: 'Reward signal teaches what gold labels cannot.'")
else:
    print("  Mixed result — interpret per-metric carefully.")

In [ ]:
# CELL 11: Save artifacts
LOGS = Path(REPO_DIR) / 'logs'
LOGS.mkdir(exist_ok=True)

# Per-row SFT
(LOGS / 'eval_sft_per_row.jsonl').write_text('\n'.join(json.dumps(r) for r in per_row) + '\n')

# Aggregate SFT
sft_path = LOGS / 'eval_sft.json'
sft_path.write_text(json.dumps({
    'sft_baseline': {
        'detection': sft_overall['detection'], 'fpr': sft_overall['fpr'],
        'precision': sft_overall['precision'], 'f1': sft_overall['f1'], 'n': sft_overall['n'],
        'per_difficulty': {d: {'n': m['n'], 'detection_rate': m['detection']} for d, m in sft_by_diff.items()},
        'threshold': THR,
    },
    'meta': {
        'base_model': BASE_MODEL,
        'lora_config': {'r': 32, 'alpha': 64, 'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']},
        'training': {'epochs': 3, 'batch_size': 4, 'grad_accum': 2, 'lr': 5e-5, 'optim': 'paged_adamw_8bit'},
        'dataset_size': len(examples),
        'temperature': 0.0, 'seed': 42,
    },
}, indent=2))

# Comparison
cmp_path = LOGS / 'sft_vs_grpo_comparison.json'
cmp_path.write_text(json.dumps({
    'sft_baseline': {k: sft_overall[k] for k in ('detection', 'fpr', 'precision', 'f1', 'n')},
    'v2_grpo': {k: GRPO_NUMBERS[k] for k in ('detection', 'fpr', 'precision', 'f1', 'n')},
    'delta_sft_minus_grpo': {
        k: sft_overall[k] - GRPO_NUMBERS[k] for k in ('detection', 'fpr', 'precision', 'f1')
    },
    'note': 'Same training corpus, same LoRA hyperparams (r=32, alpha=64), same target modules. Only the training algo differs: SFT (imitation of gold answers) vs GRPO (online RL on AnalyzerRubricV2 reward).',
}, indent=2))

print('Saved:')
print(' -', LOGS / 'eval_sft_per_row.jsonl')
print(' -', sft_path)
print(' -', cmp_path)

In [ ]:
# CELL 12: Bundle SFT LoRA + download all artifacts
import shutil

lora_zip = '/content/sft_baseline_lora.zip'
shutil.make_archive(lora_zip.replace('.zip', ''), 'zip', f'{OUTPUT_DIR}/final')
print(f'Bundled LoRA -> {lora_zip} ({Path(lora_zip).stat().st_size / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(str(LOGS / 'eval_sft_per_row.jsonl'))
    files.download(str(sft_path))
    files.download(str(cmp_path))
    files.download(lora_zip)
except ImportError:
    print('Not in Colab — files at:')
    print(' ', LOGS / 'eval_sft_per_row.jsonl')
    print(' ', sft_path)
    print(' ', cmp_path)
    print(' ', lora_zip)

## Next steps

Drop into local repo + commit:
```bash
mv ~/Downloads/eval_sft_per_row.jsonl logs/
mv ~/Downloads/eval_sft.json logs/
mv ~/Downloads/sft_vs_grpo_comparison.json logs/
mv ~/Downloads/sft_baseline_lora.zip checkpoints/  # or wherever you want
git add logs/eval_sft.json logs/eval_sft_per_row.jsonl logs/sft_vs_grpo_comparison.json
git commit -m "feat(eval): B.1 SFT baseline vs v2 GRPO comparison"
git push
```

**Cite in README:** the headline number is the **FPR ratio** (SFT FPR / GRPO FPR). If SFT FPR is significantly higher, that's the GRPO win — *"GRPO's reward shaping reduces FPR Nx vs imitation SFT, with same training data and same LoRA."*

**Cite in Q&A rehearsal:** when asked *"did GRPO actually help?"* — *"yes, measured. SFT baseline reached X% detection / Y% FPR with same data + same LoRA. GRPO is Z% detection / W% FPR. The reward signal (specifically AnalyzerRubricV2's calibration weight + FP penalty) is what delivers the asymmetric improvement."*